<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/06_modeling_workflow_with_sklearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modeling with scikit-learn

In [19]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [20]:
# standard library imports
import functools
from pathlib import Path

# data tools
import polars as pl
import polars.selectors as cs
import geopandas

# graphing tools
import altair as alt
import matplotlib.pyplot as plt

# sklearn config
from sklearn import set_config

# sklearn base types
from sklearn import tree
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

# data
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# pipeline nodes
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

# models
from sklearn.tree import DecisionTreeClassifier


In [21]:
alt.data_transformers.disable_max_rows()
set_config(transform_output='polars')

DataTransformerRegistry.enable('default')

In [23]:
iris = load_iris(as_frame=True)
iris.frame.columns
iris.target_names
iris.frame.head()

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'target'],
      dtype='object')

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [24]:
label_mapper = lambda label: iris.target_names[label]
df = (
    pl.from_pandas(iris.frame)
    .select(
        pl.col("sepal length (cm)").alias("sepalLength"),
        pl.col("sepal width (cm)").alias("sepalWidth"),
        pl.col("petal length (cm)").alias("petalLength"),
        pl.col("petal width (cm)").alias("petalWidth"),
        pl.col("target").map_elements(label_mapper).alias("species"),
    )
)
df.head()

sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.


sepalLength,sepalWidth,petalLength,petalWidth,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""setosa"""
4.9,3.0,1.4,0.2,"""setosa"""
4.7,3.2,1.3,0.2,"""setosa"""
4.6,3.1,1.5,0.2,"""setosa"""
5.0,3.6,1.4,0.2,"""setosa"""


In [27]:
alt.Chart(df).mark_point().encode(
    x=alt.X('petalWidth').title("Petal Width"),
    y=alt.Y('petalLength').title("Petal Length"),
    color=alt.Color('species').title("Species")
)

alt.Chart(...)

In [31]:
chart1 = alt.Chart(df).mark_point().encode(
    x=alt.X('petalWidth').title("Petal Width"),
    y=alt.Y('petalLength').title("Petal Length"),
    color=alt.Color('species').title("Species")
).properties(
    height=300,
    width=300
)

chart2 = alt.Chart(df).mark_bar().encode(
    x=alt.X('count()'),
    y=alt.Y('petalWidth:Q').bin(maxbins=30).title("Petal Width (binned)"),
    color=alt.Color('species').title("Species")
).properties(
    height=300,
    width=100
)

chart1 | chart2

alt.HConcatChart(...)

In [34]:
alt.Chart(df).mark_point(opacity=0.5).encode(
    alt.X(alt.repeat("column"), type='quantitative'),
    alt.Y(alt.repeat("row"), type='quantitative'),
    color='species:N'
).properties(
    width=200,
    height=200
).repeat(
    row=['petalLength', 'petalWidth'],
    column=['sepalLength', 'sepalWidth']
).interactive()

alt.RepeatChart(...)

In [35]:
alt.Chart(df, width=500).transform_window(
    index='count()'
).transform_fold(
    ['petalLength', 'petalWidth', 'sepalLength', 'sepalWidth']
).mark_line().encode(
    x='key:N',
    y='value:Q',
    color='species:N',
    detail='index:N',
    opacity=alt.value(0.5)
)

alt.Chart(...)

In [36]:
alt.Chart(df).transform_fold(
    [
        "petalWidth",
        "petalLength",
        "sepalWidth",
        "sepalLength",
    ],
    as_=["Measurement_type", "value"],
).transform_density(
    density="value",
    bandwidth=0.3,
    groupby=["Measurement_type"],
    extent=[0, 8],
).mark_area().encode(
    alt.X("value:Q"),
    alt.Y("density:Q"),
    alt.Row("Measurement_type:N"),
).properties(
    width=300, height=50
)

alt.Chart(...)